# Salary and Industry Trend Analysis

This notebook scaffolds a source-only analysis for salary visibility, industry demand, role-family movement, skill signals, geography, and time trends in the approved jobs snapshot. It is intentionally conservative: every insight must be tied to coverage, source fields, and visible limitations before interpretation.

## Scope and non-goals

**In scope:** read-only analysis of the fixed snapshot at `data/analysis/jobs_27_07_2026.csv`, using the same checksum, row count, column count, schema, date bounds, and cutoff used by the prior notebooks.

**Non-goals:** universal labor-market claims, annualized or year-over-year claims, model accuracy claims, causal salary-impact claims, destructive writes, production scraper changes, URL deduplication, or corrections to source data.

**Salary limitation:** salary fields are expected to be sparse and source-dependent. Any salary trend must first report coverage by source, currency, industry, role family, and geography. Low-coverage segments are descriptive only and must not be framed as market compensation benchmarks.

## 1. Reproducibility contract

The snapshot contract is fixed. Execution should stop if the local file, checksum, dimensions, schema order, or date coverage differs from the approved review input. This notebook reads the CSV and writes nothing.

In [1]:
import hashlib
import json
import re
from pathlib import Path

import pandas as pd

In [2]:
SNAPSHOT_RELATIVE_PATH = Path("data/analysis/jobs_27_07_2026.csv")
EXPECTED_SHA256 = "c3412e3315ee4c3df02f9a4f9c0e570747518f9e1ad70f9a6e1e86ceb260fd71"
EXPECTED_ROWS = 26_619
EXPECTED_COLUMNS = 26
SNAPSHOT_CUTOFF = pd.Timestamp("2026-07-27 23:59:59.999999999", tz="UTC")
EXPECTED_DATE_START = pd.Timestamp("2026-03-11", tz="UTC")
EXPECTED_DATE_END = pd.Timestamp("2026-07-27", tz="UTC")
EXPECTED_SCHEMA = [
    "id", "site", "job_url", "title", "company", "raw_location",
    "country", "state", "career", "scraped_by", "industry_niche",
    "is_premium", "job_type", "salary_min", "salary_max", "currency",
    "salary_source", "description", "cleaned_description", "keywords_json",
    "embedding", "date_posted", "date_scraped", "last_updated",
    "status", "country_code",
]
STRING_COLUMNS = [column for column in EXPECTED_SCHEMA if column not in {
    "salary_min", "salary_max", "is_premium"
}]
NA_TOKENS = ["", "NULL", "null", "None", "none", "NaN", "nan"]
MISSING_TEXT_VALUES = {"", "nan", "none", "null", "<na>"}
WEEK_PERIOD = "W-SUN"  # Pandas period start_time is Monday for weeks ending Sunday.
MONTH_PERIOD = "M"
MIN_SEGMENT_ROWS = 30  # Avoid interpreting tiny slices as stable trends.
MIN_SALARY_VISIBLE_ROWS = 10  # Salary summaries below this are audit-only.
MAX_REASONABLE_MONTHLY_SALARY_MXN = 500_000  # Guardrail for obvious parsing or source errors.
MIN_REASONABLE_MONTHLY_SALARY_MXN = 1  # Zero and negative values are invalid after cleaning.
TOP_N = 20

In [3]:
def locate_repo_root(start: Path) -> Path:
    """Locate the repository root from a notebook or repository working directory."""
    resolved_start = start.resolve()
    for candidate in (resolved_start, *resolved_start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(f"Repository root not found from {resolved_start}.")


def sha256_file(path: Path, chunk_size: int = 1_048_576) -> str:
    """Compute a file SHA-256 digest without loading the whole file into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [4]:
repo_root = locate_repo_root(Path.cwd())
snapshot_path = repo_root / SNAPSHOT_RELATIVE_PATH
if not snapshot_path.is_file():
    raise FileNotFoundError(f"Approved snapshot not found at {snapshot_path}.")

actual_sha256 = sha256_file(snapshot_path)
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError(
        "Snapshot checksum differs from the approved contract. "
        f"Expected {EXPECTED_SHA256}, received {actual_sha256}."
    )

jobs = pd.read_csv(
    snapshot_path,
    encoding="utf-8",
    sep=",",
    quotechar='"',
    dtype={column: "string" for column in STRING_COLUMNS},
    keep_default_na=True,
    na_values=NA_TOKENS,
    on_bad_lines="error",
    low_memory=False,
)
date_scraped = pd.to_datetime(jobs["date_scraped"], errors="coerce", utc=True)
date_posted = pd.to_datetime(jobs["date_posted"], errors="coerce", utc=True)

if jobs.shape != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise ValueError(f"Expected {(EXPECTED_ROWS, EXPECTED_COLUMNS)}, received {jobs.shape}.")
if jobs.columns.tolist() != EXPECTED_SCHEMA:
    raise ValueError("Snapshot schema or column order differs from the approved contract.")
if date_scraped.isna().any():
    raise ValueError("Snapshot contains missing or unparseable date_scraped values.")
if (
    date_scraped.min().normalize() != EXPECTED_DATE_START
    or date_scraped.max().normalize() != EXPECTED_DATE_END
    or date_scraped.gt(SNAPSHOT_CUTOFF).any()
):
    raise ValueError("Snapshot date coverage differs from the approved contract.")

contract_summary = pd.Series({
    "snapshot": str(SNAPSHOT_RELATIVE_PATH),
    "sha256": actual_sha256,
    "rows": len(jobs),
    "columns": len(jobs.columns),
    "date_scraped_start": date_scraped.min().date().isoformat(),
    "date_scraped_end": date_scraped.max().date().isoformat(),
}, name="reproducibility_contract")
contract_summary

snapshot                              data/analysis/jobs_27_07_2026.csv
sha256                c3412e3315ee4c3df02f9a4f9c0e570747518f9e1ad70f...
rows                                                              26619
columns                                                              26
date_scraped_start                                           2026-03-11
date_scraped_end                                             2026-07-27
Name: reproducibility_contract, dtype: object

## 2. Salary data quality

Salary metrics are only interpretable after visibility, validity, currency, and source coverage are visible. The guardrails below flag impossible values and outliers for review; they do not rewrite the source snapshot.

In [5]:
def normalize_text(value: object) -> str:
    """Normalize nullable text for grouping and transparent heuristics."""
    if pd.isna(value):
        return ""
    text = str(value).strip().casefold()
    if text in MISSING_TEXT_VALUES:
        return ""
    return " ".join(re.sub(r"[^a-z0-9]+", " ", text).split())


def coverage_table(frame: pd.DataFrame, group_columns: list[str], value_column: str) -> pd.DataFrame:
    """Summarize non-null coverage for one value column by reviewable groups."""
    grouped = frame.groupby(group_columns, dropna=False, sort=True)
    summary = grouped.agg(
        rows=(value_column, "size"),
        visible_rows=(value_column, lambda values: int(values.notna().sum())),
    ).reset_index()
    summary["coverage_rate"] = summary["visible_rows"] / summary["rows"]
    return summary.sort_values(["visible_rows", "rows"], ascending=[False, False])


salary_min = pd.to_numeric(jobs["salary_min"], errors="coerce")
salary_max = pd.to_numeric(jobs["salary_max"], errors="coerce")
salary_visible = salary_min.notna() | salary_max.notna()
salary_pair_valid = (
    salary_min.notna()
    & salary_max.notna()
    & salary_min.ge(MIN_REASONABLE_MONTHLY_SALARY_MXN)
    & salary_max.ge(MIN_REASONABLE_MONTHLY_SALARY_MXN)
    & salary_min.le(salary_max)
)
salary_midpoint = ((salary_min + salary_max) / 2).where(salary_pair_valid)
salary_outlier_guardrail = salary_pair_valid & salary_midpoint.gt(MAX_REASONABLE_MONTHLY_SALARY_MXN)

analysis_jobs = jobs.assign(
    row_number=pd.RangeIndex(len(jobs)),
    title_norm=jobs["title"].map(normalize_text).astype("string"),
    industry_niche_norm=jobs["industry_niche"].map(normalize_text).astype("string"),
    career_norm=jobs["career"].map(normalize_text).astype("string"),
    job_type_norm=jobs["job_type"].map(normalize_text).astype("string"),
    site_norm=jobs["site"].map(normalize_text).astype("string"),
    country_norm=jobs["country"].map(normalize_text).astype("string"),
    state_norm=jobs["state"].map(normalize_text).astype("string"),
    raw_location_norm=jobs["raw_location"].map(normalize_text).astype("string"),
    date_posted_parsed=date_posted,
    date_scraped_parsed=date_scraped,
    salary_min_numeric=salary_min,
    salary_max_numeric=salary_max,
    salary_visible=salary_visible,
    salary_pair_valid=salary_pair_valid,
    salary_midpoint=salary_midpoint,
    salary_outlier_guardrail=salary_outlier_guardrail,
)

salary_quality_summary = pd.Series({
    "rows": len(analysis_jobs),
    "salary_visible_rows": int(analysis_jobs["salary_visible"].sum()),
    "valid_salary_pair_rows": int(analysis_jobs["salary_pair_valid"].sum()),
    "invalid_visible_salary_rows": int((analysis_jobs["salary_visible"] & ~analysis_jobs["salary_pair_valid"]).sum()),
    "outlier_guardrail_rows": int(analysis_jobs["salary_outlier_guardrail"].sum()),
}, name="salary_quality_summary")
salary_quality_by_source = coverage_table(analysis_jobs, ["site", "currency", "salary_source"], "salary_midpoint")
{
    "summary": salary_quality_summary,
    "by_source": salary_quality_by_source.head(TOP_N),
}

{'summary': rows                           26619
 salary_visible_rows             8932
 valid_salary_pair_rows          8931
 invalid_visible_salary_rows        1
 outlier_guardrail_rows            51
 Name: salary_quality_summary, dtype: int64,
 'by_source':            site currency salary_source   rows  visible_rows  coverage_rate
 0  computrabajo      MXN          <NA>   9461          5357       0.566219
 3           occ      MXN          <NA>   3044          1825       0.599540
 1        indeed      MXN          <NA>  10127          1749       0.172707
 2      linkedin      MXN          <NA>   3987             0       0.000000}

## 3. Industry demand

Demand here means listing volume in the snapshot, not total market demand. Groupings stay source-visible so downstream claims can separate industry, career, job type, site, and date effects.

In [6]:
industry_dimensions = ["industry_niche", "career", "job_type", "site"]
industry_demand = analysis_jobs.groupby(industry_dimensions, dropna=False, sort=True).agg(
    listings=("id", "size"),
    salary_visible_rows=("salary_visible", "sum"),
    valid_salary_rows=("salary_pair_valid", "sum"),
    first_date_scraped=("date_scraped_parsed", "min"),
    last_date_scraped=("date_scraped_parsed", "max"),
    first_date_posted=("date_posted_parsed", "min"),
    last_date_posted=("date_posted_parsed", "max"),
).reset_index()
industry_demand["salary_visibility_rate"] = (
    industry_demand["salary_visible_rows"] / industry_demand["listings"]
)
industry_demand["interpretation_ready"] = (
    industry_demand["listings"].ge(MIN_SEGMENT_ROWS)
    & industry_demand["valid_salary_rows"].ge(MIN_SALARY_VISIBLE_ROWS)
)
industry_demand.sort_values(["listings", "salary_visible_rows"], ascending=[False, False]).head(TOP_N)

,industry_niche,career,job_type,site,listings,salary_visible_rows,valid_salary_rows,first_date_scraped,last_date_scraped,first_date_posted,last_date_posted,salary_visibility_rate,interpretation_ready
177,Ingeniería de Datos,Data Analyst,presencial,indeed,1659,108,107,2026-03-18 23:53:17.367688+00:00,2026-07-26 14:01:53.289321+00:00,2021-09-06 00:00:00+00:00,2026-07-25 00:00:00+00:00,0.065099,True
182,Ingeniería de Datos,Data Analyst,<NA>,computrabajo,1643,1146,1146,2026-05-10 13:58:35.781728+00:00,2026-07-26 14:01:59.789375+00:00,NaT,NaT,0.697505,True
135,Infraestructura Cloud,Cloud Engineer,presencial,indeed,1632,173,173,2026-03-22 05:07:44.419040+00:00,2026-07-27 03:44:50.811213+00:00,2025-07-16 00:00:00+00:00,2026-07-26 00:00:00+00:00,0.106005,True
8,Analítica de Datos,Business Intelligence,<NA>,computrabajo,1151,820,820,2026-05-10 14:02:27.067356+00:00,2026-07-25 14:08:57.970474+00:00,NaT,NaT,0.712424,True
41,Desarrollo de Software,Python Developer,<NA>,computrabajo,995,626,626,2026-05-10 14:00:46.025792+00:00,2026-07-26 14:03:41.173830+00:00,NaT,NaT,0.629146,True
168,Ingeniería de Datos,Analytics Engineer,<NA>,occ,864,492,492,2026-05-10 13:59:30.244242+00:00,2026-07-25 14:06:31.016845+00:00,NaT,NaT,0.569444,True
3,Analítica de Datos,Business Intelligence,presencial,indeed,861,125,125,2026-05-10 14:02:16.558080+00:00,2026-07-26 14:04:56.371649+00:00,2025-10-15 00:00:00+00:00,2026-07-25 00:00:00+00:00,0.145180,True
192,Ingeniería de Datos,Data Engineer,<NA>,computrabajo,823,406,406,2026-05-10 13:57:48.927386+00:00,2026-07-25 14:05:28.159616+00:00,NaT,NaT,0.493317,True
9,Analítica de Datos,Business Intelligence,<NA>,occ,588,373,373,2026-05-10 14:02:46.582819+00:00,2026-07-26 14:05:07.145585+00:00,NaT,NaT,0.634354,True
51,Desarrollo de Software,React Developer,<NA>,computrabajo,524,334,334,2026-05-10 14:01:22.121137+00:00,2026-07-26 14:04:11.086362+00:00,NaT,NaT,0.637405,True


## 4. Role-family heuristics

Role families are conservative title buckets, not a model. Rules are ordered, deterministic, and auditable. Ambiguous titles fall into `other` rather than forcing precision the data does not support.

In [7]:
ROLE_BUCKET_RULES = [
    ("data analyst", re.compile(r"\b(data analyst|analista de datos|business intelligence|bi analyst|power bi)\b")),
    ("data engineer", re.compile(r"\b(data engineer|ingeniero de datos|etl|data pipeline|big data)\b")),
    ("backend", re.compile(r"\b(backend|back end|api developer|java developer|python developer|node developer)\b")),
    ("frontend", re.compile(r"\b(frontend|front end|react|angular|vue|ui developer)\b")),
    ("fullstack", re.compile(r"\b(fullstack|full stack|desarrollador full)\b")),
    ("devops/cloud", re.compile(r"\b(devops|sre|site reliability|cloud|aws|azure|kubernetes|docker)\b")),
    ("industrial/maintenance", re.compile(r"\b(maintenance|mantenimiento|industrial|manufactur|production|plc|calidad|quality)\b")),
]
ROLE_BUCKETS = [bucket for bucket, _ in ROLE_BUCKET_RULES] + ["other"]


def classify_role_family(title_norm: str) -> str:
    """Classify a normalized title with deterministic, conservative rules."""
    title = "" if pd.isna(title_norm) else str(title_norm)
    for bucket, pattern in ROLE_BUCKET_RULES:
        if pattern.search(title):
            return bucket
    return "other"


analysis_jobs = analysis_jobs.assign(
    role_family=analysis_jobs["title_norm"].map(classify_role_family).astype("string")
)
role_family_summary = analysis_jobs.groupby("role_family", dropna=False, sort=True).agg(
    listings=("id", "size"),
    salary_visible_rows=("salary_visible", "sum"),
    valid_salary_rows=("salary_pair_valid", "sum"),
    median_salary_midpoint=("salary_midpoint", "median"),
).reset_index()
role_family_summary["salary_visibility_rate"] = (
    role_family_summary["salary_visible_rows"] / role_family_summary["listings"]
)
role_family_summary["interpretation_ready"] = (
    role_family_summary["listings"].ge(MIN_SEGMENT_ROWS)
    & role_family_summary["valid_salary_rows"].ge(MIN_SALARY_VISIBLE_ROWS)
)
role_family_summary.sort_values("listings", ascending=False)

,role_family,listings,salary_visible_rows,valid_salary_rows,median_salary_midpoint,salary_visibility_rate,interpretation_ready
7,other,15190,5084,5083,23000.0,0.334694,True
1,data analyst,2872,1484,1484,18000.0,0.516713,True
2,data engineer,2158,597,597,42000.0,0.276645,True
3,devops/cloud,2033,355,355,52500.0,0.174619,True
0,backend,1182,224,224,35750.0,0.189509,True
4,frontend,1087,322,322,35000.0,0.296228,True
6,industrial/maintenance,1056,424,424,14999.5,0.401515,True
5,fullstack,1041,442,442,36250.0,0.424592,True


## 5. Skill signal

Skill signals come from `keywords_json` when parseable and from cleaned descriptions only as a fallback text surface. Frequency and co-occurrence can support descriptive hypotheses, but this notebook must not claim causal salary impact.

In [8]:
def parse_keywords_json(value: object) -> list[str]:
    """Parse keywords_json into normalized string tokens without failing the notebook."""
    if pd.isna(value) or str(value).strip().casefold() in MISSING_TEXT_VALUES:
        return []
    try:
        parsed = json.loads(str(value))
    except json.JSONDecodeError:
        return []
    if isinstance(parsed, list):
        return [normalize_text(item) for item in parsed if normalize_text(item)]
    if isinstance(parsed, dict):
        values = []
        for item in parsed.values():
            if isinstance(item, list):
                values.extend(item)
            else:
                values.append(item)
        return [normalize_text(item) for item in values if normalize_text(item)]
    return []


keyword_lists = analysis_jobs["keywords_json"].map(parse_keywords_json)
keyword_rows = [
    {"row_number": row_number, "keyword": keyword}
    for row_number, keywords in zip(analysis_jobs["row_number"], keyword_lists)
    for keyword in sorted(set(keywords))
]
keyword_frame = pd.DataFrame(keyword_rows, columns=["row_number", "keyword"])
keyword_frequency = keyword_frame.groupby("keyword", sort=True).agg(
    rows=("row_number", "nunique"),
).reset_index().sort_values(["rows", "keyword"], ascending=[False, True])
keyword_role_frequency = keyword_frame.merge(
    analysis_jobs[["row_number", "role_family", "salary_pair_valid", "salary_midpoint"]],
    on="row_number",
    how="left",
).groupby(["role_family", "keyword"], sort=True).agg(
    rows=("row_number", "nunique"),
    valid_salary_rows=("salary_pair_valid", "sum"),
    median_salary_midpoint=("salary_midpoint", "median"),
).reset_index()
{
    "keyword_frequency": keyword_frequency.head(TOP_N),
    "keyword_role_frequency": keyword_role_frequency.sort_values("rows", ascending=False).head(TOP_N),
}

{'keyword_frequency':         keyword  rows
 3318       team   908
 771     backend   837
 1253      datos   656
 854       build   519
 3078   security   498
 16            1   476
 3254     strong   410
 3390       time   410
 3012       role   398
 3132   services   396
 2762    product   367
 796       bases   347
 2715        ppg   324
 1571     equipo   315
 2665        php   297
 1494   employee   280
 3600  workflows   279
 504      across   274
 2669  pipelines   272
 3047   scalable   272,
 'keyword_role_frequency':       role_family    keyword  rows  valid_salary_rows  median_salary_midpoint
 11332       other       team   657                109                 27500.0
 8908        other    backend   413                101                 25000.0
 9368        other      datos   362                103                 22800.0
 8989        other      build   358                 82                 26250.0
 11103       other   security   345                 64                 250

## 6. Geography

Geographic summaries keep `country`, `state`, and `raw_location` separate because source normalization may be incomplete. Salary visibility by location must be reported before interpreting regional compensation differences.

In [9]:
geography_coverage = analysis_jobs.groupby(
    ["country", "state", "raw_location"], dropna=False, sort=True
).agg(
    listings=("id", "size"),
    sites=("site", "nunique"),
    salary_visible_rows=("salary_visible", "sum"),
    valid_salary_rows=("salary_pair_valid", "sum"),
    median_salary_midpoint=("salary_midpoint", "median"),
).reset_index()
geography_coverage["salary_visibility_rate"] = (
    geography_coverage["salary_visible_rows"] / geography_coverage["listings"]
)
geography_coverage["interpretation_ready"] = (
    geography_coverage["listings"].ge(MIN_SEGMENT_ROWS)
    & geography_coverage["valid_salary_rows"].ge(MIN_SALARY_VISIBLE_ROWS)
)
geography_coverage.sort_values(["listings", "salary_visible_rows"], ascending=[False, False]).head(TOP_N)

,country,state,raw_location,listings,sites,salary_visible_rows,valid_salary_rows,median_salary_midpoint,salary_visibility_rate,interpretation_ready
297,México,<NA>,"Ciudad de México, DIF, MX",2202,1,212,212,42500.0,0.096276,True
416,México,<NA>,"Desde casa, MX",1427,1,247,247,43900.0,0.173090,True
183,México,<NA>,"Benito Juárez, Ciudad de México DF",1338,1,716,716,40000.0,0.535127,True
807,México,<NA>,"Miguel Hidalgo, Ciudad de México DF",939,1,504,504,25000.0,0.536741,True
296,México,<NA>,Ciudad de México,844,1,472,472,35000.0,0.559242,True
58,México,Jalisco,"Guadalajara, JAL, MX",844,1,48,48,50000.0,0.056872,True
1443,México,<NA>,<NA>,832,1,0,0,NaN,0.000000,False
833,México,<NA>,"Monterrey, Nuevo León",701,2,405,405,37500.0,0.577746,True
59,México,Jalisco,"Guadalajara, Jalisco",587,2,394,394,35000.0,0.671210,True
832,México,<NA>,"Monterrey, NLE, MX",525,1,52,52,40000.0,0.099048,True


## 7. Time trends

`date_posted` and `date_scraped` answer different questions. `date_scraped` describes snapshot collection timing; `date_posted` describes source-declared listing age when available. Weekly and monthly trends must show both where possible.

In [10]:
analysis_jobs = analysis_jobs.assign(
    date_scraped_week=analysis_jobs["date_scraped_parsed"].dt.to_period(WEEK_PERIOD).dt.start_time,
    date_scraped_month=analysis_jobs["date_scraped_parsed"].dt.to_period(MONTH_PERIOD).dt.start_time,
    date_posted_week=analysis_jobs["date_posted_parsed"].dt.to_period(WEEK_PERIOD).dt.start_time,
    date_posted_month=analysis_jobs["date_posted_parsed"].dt.to_period(MONTH_PERIOD).dt.start_time,
)
weekly_scraped_volume = analysis_jobs.groupby("date_scraped_week", dropna=False, sort=True).agg(
    listings=("id", "size"),
    salary_visible_rows=("salary_visible", "sum"),
    valid_salary_rows=("salary_pair_valid", "sum"),
).reset_index()
monthly_posted_volume = analysis_jobs.groupby("date_posted_month", dropna=False, sort=True).agg(
    listings=("id", "size"),
    salary_visible_rows=("salary_visible", "sum"),
    valid_salary_rows=("salary_pair_valid", "sum"),
).reset_index()
role_month_movement = analysis_jobs.groupby(
    ["date_scraped_month", "industry_niche", "role_family"], dropna=False, sort=True
).agg(
    listings=("id", "size"),
    salary_visible_rows=("salary_visible", "sum"),
    valid_salary_rows=("salary_pair_valid", "sum"),
).reset_index()
{
    "weekly_scraped_volume": weekly_scraped_volume,
    "monthly_posted_volume": monthly_posted_volume,
    "role_month_movement": role_month_movement.sort_values("listings", ascending=False).head(TOP_N),
}

/tmp/ipykernel_66154/562524513.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  date_scraped_week=analysis_jobs["date_scraped_parsed"].dt.to_period(WEEK_PERIOD).dt.start_time,
/tmp/ipykernel_66154/562524513.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  date_scraped_month=analysis_jobs["date_scraped_parsed"].dt.to_period(MONTH_PERIOD).dt.start_time,
/tmp/ipykernel_66154/562524513.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  date_posted_week=analysis_jobs["date_posted_parsed"].dt.to_period(WEEK_PERIOD).dt.start_time,
/tmp/ipykernel_66154/562524513.py:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  date_posted_month=analysis_jobs["date_posted_parsed"].dt.to_period(MONTH_PERIOD).dt.start_time,


{'weekly_scraped_volume':    date_scraped_week  listings  salary_visible_rows  valid_salary_rows
 0         2026-03-09        92                   28                 28
 1         2026-03-16       678                  141                141
 2         2026-03-23       816                   97                 97
 3         2026-03-30       337                   40                 40
 4         2026-04-06       598                   70                 70
 5         2026-04-13       359                   67                 67
 6         2026-04-20       440                   72                 72
 7         2026-04-27       506                   79                 79
 8         2026-05-04      1635                  672                672
 9         2026-05-11      2571                  933                933
 10        2026-05-18      2424                  893                893
 11        2026-05-25      2565                  985                985
 12        2026-06-01      2336        

## 8. Validation assertions

These assertions are source-only guardrails for future execution. They verify the snapshot contract, non-negative salary values after cleaning, deterministic role buckets, and absence of hidden write calls in this notebook source.

In [11]:
FORBIDDEN_WRITE_PATTERNS = (
    r"\.to_csv\s*\(",
    r"\.to_parquet\s*\(",
    r"\.to_excel\s*\(",
    r"\.to_json\s*\(",
    r"open\s*\([^\)]*[\"']w[\"']",
    r"Path\([^\)]*\)\.write_",
)


def notebook_code_source(path: Path) -> str:
    """Return concatenated code-cell source from a notebook file."""
    notebook = json.loads(path.read_text(encoding="utf-8"))
    code_cells = [cell for cell in notebook["cells"] if cell["cell_type"] == "code"]
    return "\n".join("".join(cell["source"]) for cell in code_cells)


assert jobs.shape == (EXPECTED_ROWS, EXPECTED_COLUMNS)
assert jobs.columns.tolist() == EXPECTED_SCHEMA
assert date_scraped.min().normalize() == EXPECTED_DATE_START
assert date_scraped.max().normalize() == EXPECTED_DATE_END
assert not date_scraped.gt(SNAPSHOT_CUTOFF).any()
assert not analysis_jobs.loc[analysis_jobs["salary_pair_valid"], "salary_min_numeric"].lt(0).any()
assert not analysis_jobs.loc[analysis_jobs["salary_pair_valid"], "salary_max_numeric"].lt(0).any()
assert len(ROLE_BUCKETS) == len(set(ROLE_BUCKETS))
assert classify_role_family("data analyst") == "data analyst"
assert classify_role_family("senior backend engineer") == "backend"
assert classify_role_family("unmapped title") == "other"

notebook_path = repo_root / "notebooks" / "03_salary_industry_trends.ipynb"
source_text = notebook_code_source(notebook_path)
for forbidden_pattern in FORBIDDEN_WRITE_PATTERNS:
    assert not re.search(forbidden_pattern, source_text), forbidden_pattern

## 9. Insight scorecard

Use this section after execution to separate evidence strength from speculation. Do not fill it with claims until the coverage tables above have been inspected.

| Category | Evidence standard | Findings to record after execution |
|---|---|---|
| Strongest findings | Large denominator, clear coverage, stable source mix, and consistent direction across relevant slices. | Pending execution. |
| Weak findings | Small denominator, sparse salary visibility, source-specific artifacts, or movement visible only in one slice. | Pending execution. |
| Cannot be concluded | Universal market pay, causal skill salary impact, model accuracy, employer-level compensation policy, or annual trends. | Pending execution. |

## Next step

Rerun this notebook only after placing the expected snapshot locally at `data/analysis/jobs_27_07_2026.csv`. Inspect salary coverage and source mix before interpreting any industry, role, geography, skill, or time trend.